In [1]:
# ✅ Step 1: Unzip dataset (already uploaded in Colab)
import zipfile
import os

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall()

# ✅ Step 2: Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ✅ Step 3: Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ✅ Step 4: Transform
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ✅ Step 5: Load dataset
data_dir = "PetImages"
dataset = datasets.ImageFolder(data_dir, transform=transform)


# ✅ Step 7: Dataset split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=64, num_workers=0)

# ✅ Step 8: Extended AlexNet with 2 more CNN layers
class ExtendedAlexNet(nn.Module):
    def __init__(self, num_classes=2):
        super(ExtendedAlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            # 🔥 Extra CNN Layer 1
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # 🔥 Extra CNN Layer 2
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 6))  # keep 6x6 for classifier input
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(64 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# ✅ Step 9: Model, Loss, Optimizer
model = ExtendedAlexNet(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# ✅ Step 10: Train function
def train(model, loader):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ✅ Step 11: Eval function
def evaluate(model, loader):
    model.eval()
    correct = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, preds = torch.max(out, 1)
            correct += (preds == y).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return correct / len(loader.dataset), all_preds, all_labels

# ✅ Step 12: Training loop
for epoch in range(10):
    loss = train(model, train_loader)
    acc, _, _ = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Train Loss = {loss:.4f}, Val Acc = {acc:.4f}")

# ✅ Step 13: Final testing
test_acc, preds, labels = evaluate(model, test_loader)
print(f"\nTest Accuracy = {test_acc:.4f}")
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=dataset.classes))


Device: cuda


/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1: Train Loss = 0.6585, Val Acc = 0.6762
Epoch 2: Train Loss = 0.5914, Val Acc = 0.7538
Epoch 3: Train Loss = 0.4948, Val Acc = 0.8037
Epoch 4: Train Loss = 0.4120, Val Acc = 0.7671
Epoch 5: Train Loss = 0.3456, Val Acc = 0.8568
Epoch 6: Train Loss = 0.2877, Val Acc = 0.8613
Epoch 7: Train Loss = 0.2424, Val Acc = 0.8778
Epoch 8: Train Loss = 0.2018, Val Acc = 0.8784
Epoch 9: Train Loss = 0.1672, Val Acc = 0.8834
Epoch 10: Train Loss = 0.1263, Val Acc = 0.8941

Test Accuracy = 0.8990

Classification Report:

              precision    recall  f1-score   support

         Cat       0.91      0.88      0.90      1855
         Dog       0.89      0.92      0.90      1896

    accuracy                           0.90      3751
   macro avg       0.90      0.90      0.90      3751
weighted avg       0.90      0.90      0.90      3751

